In [ ]:
import lightkurve as lk
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from astropy.timeseries import LombScargle
import astropy.units as u
import os

In [ ]:
i = 0

base_dir = "kepler_fits"
os.makedirs(base_dir, exist_ok=True)
target_list = pd.read_csv("40daylimit_periods.csv")

for idx, row in target_list.iterrows():
    kepler_id = row["Kepler ID"]

    i=i+1
    print(i)

    if pd.isna(kepler_id):
        continue

    try: 
        kic_int = int(kepler_id)
        star_folder = os.path.join(base_dir, f"KIC_{kic_int}")
        os.makedirs(star_folder, exist_ok=True)

        search_result = lk.search_lightcurve(f"KIC {kic_int}", author='Kepler')

        for lc_idx, lc_entry in enumerate(search_result):
            filename = lc_entry.productFilename
            lc_entry.download(download_dir=star_folder)

            full_path = os.path.join(star_folder, filename)

            try:
                quarter = lc_entry.table['quarter'][0]
            except:
                quarter = None
    
            target_list.at[idx, f"LC{lc_idx+1} Quarter"] = quarter
            target_list.at[idx, f"LC{lc_idx+1} Filename"] = filename
            target_list.at[idx, f"LC{lc_idx+1} Path"] = full_path
    
    except Exception as e:
        print(f"⚠️ Skipping KIC {kepler_id}: {e}")
        continue

# Save updated CSV
target_list.to_csv("fits_path_names.csv", index=False)